## Lab 21 – Secure Views and Stored Procedures
**Snowflake Fundamentals 4-day class Lab · © 2026 Innovation In Software Corporation. All rights reserved.**

Topics covered:
1. Creating a Secure View over an external stage
2. Creating a Secure Stored Procedure with EXECUTE AS OWNER
3. DDL protection — GET_DDL returns empty for non-owners
4. Secondary roles — USE SECONDARY ROLES NONE
5. Calling secure procedures with CALL and TABLE() syntax
6. Granting access to a test role

### DEMO 1 | Context Setup

> **[INSTRUCTOR NOTE]**
> The `WEATHER` database contains NYC weather JSON files hosted in a public S3 bucket provided by Snowflake. A FILE FORMAT and external STAGE are created to point at this data. No data is loaded into native tables — the secure view queries the stage directly.

In [ ]:
%%sql -r dataframe_1
USE ROLE accountadmin;

CREATE DATABASE IF NOT EXISTS weather;

USE SCHEMA WEATHER.PUBLIC;

In [ ]:
%%sql -r dataframe_2
-- Create FILE FORMAT
CREATE FILE FORMAT IF NOT EXISTS json_format
TYPE = 'JSON';

-- Create external stage
CREATE STAGE IF NOT EXISTS nyc_weather
URL = 's3://snowflake-workshop-lab/weather-nyc';

### DEMO 2 | Create the Secure View

> **[INSTRUCTOR NOTE]**
> `CREATE SECURE VIEW` hides the view definition from non-owners — `GET_DDL` and `SHOW VIEWS` will not return the SQL body for users who do not own the view. This protects the stage URL and query logic from reverse-engineering.

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE SECURE VIEW vw_weather AS
SELECT metadata$filename         AS filename,
       metadata$file_row_number  AS rnum,
       TO_VARIANT($1)            AS v
FROM @nyc_weather (file_format => 'json_format');

### DEMO 3 | Create the Secure Stored Procedure

> **[INSTRUCTOR NOTE]**
> `EXECUTE AS OWNER` means the procedure runs with the owner's privileges, not the caller's. The caller (test_role) does not need direct access to the stage — they only need USAGE on the procedure. This is the principle of least privilege in action.

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE SECURE PROCEDURE get_weather_data()
RETURNS TABLE (filename VARCHAR, rnum NUMBER, v VARIANT)
LANGUAGE SQL
EXECUTE AS OWNER
AS
$$
DECLARE
    res RESULTSET;
BEGIN
    res := (SELECT metadata$filename         AS filename,
                   metadata$file_row_number  AS rnum,
                   TO_VARIANT($1)            AS v
            FROM @nyc_weather (file_format => 'json_format'));
    RETURN TABLE(res);
END;
$$;

In [ ]:
%%sql -r dataframe_5
-- Verify the view DDL is visible to the owner
SHOW VIEWS LIKE 'VW_WEATHER'
->> SELECT "name", "is_secure", "text" FROM $1;

In [ ]:
%%sql -r dataframe_6
-- Verify the procedure DDL is visible to the owner
DESC PROCEDURE GET_WEATHER_DATA()
->> SHOW PROCEDURES LIKE 'GET_WEATHER_DATA'
->> SELECT 'Name'      AS key, "name"      AS value FROM $1 UNION ALL
    SELECT 'Is Secure',        "is_secure"           FROM $1 UNION ALL
    SELECT 'Body',             "value"               FROM $2 WHERE "property" = 'body';

### DEMO 4 | Create a Test Role and Grant Privileges

> **[INSTRUCTOR NOTE]**
> `test_role` is granted SELECT on the view and USAGE on the procedure, but NOT direct access to the stage or the underlying files. This models a real-world scenario where an analyst role is given access to curated objects only.

In [ ]:
%%sql -r dataframe_7
CREATE OR REPLACE ROLE test_role;

GRANT SELECT ON VIEW vw_weather TO ROLE test_role;
GRANT USAGE ON PROCEDURE get_weather_data() TO ROLE test_role;

GRANT USAGE ON DATABASE WEATHER TO ROLE test_role;
GRANT USAGE ON SCHEMA WEATHER.PUBLIC TO ROLE test_role;

-- Grant test_role to the current user
BEGIN
    EXECUTE IMMEDIATE 'GRANT ROLE test_role TO USER ' || CURRENT_USER();
END;

### DEMO 5 | Demonstrate DDL Protection

> **[INSTRUCTOR NOTE]**
> Switch to `test_role` and show that `GET_DDL` raises an error for both the secure view and the secure procedure. `USE SECONDARY ROLES NONE` is critical here — without it, the ACCOUNTADMIN secondary role would still be active and would bypass the protection.

#### What Are Secondary Roles?

In Snowflake a user can have multiple roles granted to them. When you run `USE ROLE`, the specified role becomes the **primary role**. Secondary roles are additional roles that supplement the primary role's privileges.

`USE SECONDARY ROLES NONE` disables all secondary roles for the session, limiting privileges strictly to the primary role. This is required here to accurately demonstrate the DDL protection behaviour of secure objects.

In [ ]:
%%sql -r dataframe_8
USE ROLE test_role;
USE SECONDARY ROLES NONE;
USE WAREHOUSE SNOWFLAKE_LEARNING_WH;

In [ ]:
%%sql -r dataframe_9
-- Check secure view DDL (should be hidden / raise error)
SELECT GET_DDL('VIEW', 'WEATHER.PUBLIC.VW_WEATHER');

In [ ]:
%%sql -r dataframe_10
SHOW VIEWS LIKE 'VW_WEATHER'
->> SELECT "name", "is_secure", "text" FROM $1;

In [ ]:
%%sql -r dataframe_11
-- Check secure procedure DDL (should be hidden / raise error)
SELECT GET_DDL('PROCEDURE', 'WEATHER.PUBLIC.GET_WEATHER_DATA()');

In [ ]:
%%sql -r dataframe_12
-- Demonstrate graceful error handling with GET_DDL on a secure procedure
EXECUTE IMMEDIATE $$
BEGIN
    SELECT GET_DDL('PROCEDURE', 'WEATHER.PUBLIC.GET_WEATHER_DATA()') AS ddl;
EXCEPTION
    WHEN OTHER THEN
        LET LINE := SQLCODE || ': ' || SQLERRM;
        RETURN LINE;
END;
$$;

In [ ]:
%%sql -r dataframe_13
DESC PROCEDURE GET_WEATHER_DATA()
->> SHOW PROCEDURES LIKE 'GET_WEATHER_DATA'
->> SELECT 'Name'      AS key, "name"      AS value FROM $1 UNION ALL
    SELECT 'Is Secure',        "is_secure"           FROM $1 UNION ALL
    SELECT 'Body',             "value"               FROM $2 WHERE "property" = 'body';

In [ ]:
%%sql -r dataframe_14
DESC PROCEDURE GET_WEATHER_DATA()
->> SHOW PROCEDURES LIKE 'GET_WEATHER_DATA'
->> WITH b AS (
        SELECT "value"
        FROM $2
        WHERE "property" = 'body'
    )
    SELECT a."name", a."is_secure", b."value" AS "body"
    FROM $1 AS a, b;

### DEMO 6 | Query the Secure Objects

> **[INSTRUCTOR NOTE]**
> Even though `test_role` cannot see the DDL, it can still SELECT from the view and call the procedure. The data is accessible — the body is not. This is the key takeaway: secure objects provide **DDL protection**, not data access control.

In [ ]:
%%sql -r dataframe_15
USE ROLE test_role;
USE SECONDARY ROLES NONE;

-- Query the secure view
SELECT * FROM vw_weather LIMIT 2;

In [ ]:
%%sql -r dataframe_16
-- Call the secure procedure
CALL get_weather_data()
->> SELECT * FROM $1 LIMIT 2;

In [ ]:
%%sql -r dataframe_17
-- Use TABLE() syntax
SELECT * FROM TABLE(get_weather_data()) LIMIT 2;

### Key Takeaways

- **DDL Protection:** Secure views and procedures prevent non-owners from viewing the underlying query logic via `GET_DDL` or `DESCRIBE`, reducing risks like reverse-engineering or inference attacks.
- **EXECUTE AS OWNER:** The procedure runs with the owner's privileges — callers do not need direct access to underlying objects (e.g., the stage).
- **Best Practices:** Always use `SECURE` for objects that expose sensitive logic or proprietary business rules.

### DEMO CLEANUP

> Drop objects created during the demo.

In [ ]:
%%sql -r dataframe_18
USE ROLE ACCOUNTADMIN;

-- DROP VIEW      IF EXISTS WEATHER.PUBLIC.vw_weather;
-- DROP PROCEDURE IF EXISTS WEATHER.PUBLIC.get_weather_data();
-- DROP ROLE      IF EXISTS test_role;